In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import sqlite3

df = pd.read_excel('/content/drive/MyDrive/Uber/Updated_uber-data.xlsx')
print("✅ File loaded! Shape:", df.shape)
df.head()

Mounted at /content/drive
✅ File loaded! Shape: (6745, 8)


,Request id,Pickup point,Driver id,Status,Request timestamp,Drop timestamp,Hour,Time of Day
0,619,Airport,1.0,Trip Completed,2016-07-11 11:51:00,2016-07-11 13:00:00,11,Morning
1,867,Airport,1.0,Trip Completed,2016-07-11 17:57:00,2016-07-11 18:47:00,17,Evening
2,1807,City,1.0,Trip Completed,2016-07-12 09:17:00,2016-07-12 09:58:00,9,Morning
3,2532,Airport,1.0,Trip Completed,2016-07-12 21:08:00,2016-07-12 22:03:00,21,Late Night
4,3112,City,1.0,Trip Completed,2016-07-13 08:33:16,2016-07-13 09:25:47,8,Early Morning


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
import os
os.listdir('/content/drive/MyDrive/Uber')

['Updated_uber-data.xlsx']

In [7]:
# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Create SQLite database
conn = sqlite3.connect('uber.db')
df.to_sql('trips', conn, if_exists='replace', index=False)

# Save database to Drive (safe forever!)
import shutil
shutil.copy('uber.db', '/content/drive/MyDrive/Uber/uber.db')

print("✅ Database ready and saved to Drive!")

✅ Database ready and saved to Drive!


In [12]:
def run_query(sql, title=""):
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print(f"{'='*50}")
    display(result)
    return result

In [14]:
run_query("""
    SELECT
        Status,
        COUNT(*) AS Total_Trips
    FROM trips
    GROUP BY Status
    ORDER BY Total_Trips DESC
""", title="Query 1 - Trip Count by Status");


  Query 1 - Trip Count by Status


,Status,Total_Trips
0,Trip Completed,2831
1,No Cars Available,2650
2,Cancelled,1264


Insight - Out of 6745 total trip requests, only 4031 (59.8%) were successfully completed. A significant 2650 requests (39.3%) failed due to no cars being available, and 1264 trips (18.7%) were cancelled by drivers. This means 40.2% of total demand went unfulfilled which indicates a major supply gap in Uber's operations.

In [16]:
run_query("""
    SELECT
        Hour,
        COUNT(*) AS Unfulfilled_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Hour
    ORDER BY Hour
""", title="Query 2 - Unfulfilled Requests Per Hour");


  Query 2 - Unfulfilled Requests Per Hour


,Hour,Unfulfilled_Requests
0,0,59
1,1,60
2,2,62
3,3,58
4,4,125
5,5,260
6,6,231
7,7,232
8,8,268
9,9,258


Insight - Hours 5,6,7,8,9 (Early Morning) and 17,18,19,20,21 (Evening/Night) have the highest unfulfilled requests — these are the peak problem hours for Uber.

In [17]:
run_query("""
    SELECT
        Time_of_Day,
        COUNT(*) AS Unfulfilled_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Time_of_Day
    ORDER BY Unfulfilled_Requests DESC
""", title="Query 3 - Supply Demand Gap by Time of Day");


  Query 3 - Supply Demand Gap by Time of Day


,Time_of_Day,Unfulfilled_Requests
0,Evening,1251
1,Early Morning,991
2,Late Night,548
3,Morning,441
4,Night,364
5,Afternoon,319


Insight - Night time has the highest gap (~1100 unfulfilled requests), followed by Early Morning and Morning. Afternoon has the least problems — drivers are most available during daytime.

In [18]:
run_query("""
    SELECT
        Pickup_point,
        Status,
        COUNT(*) AS Count
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Pickup_point, Status
    ORDER BY Pickup_point, Count DESC
""", title="Query 4 - Airport vs City Cancelled and No Cars");


  Query 4 - Airport vs City Cancelled and No Cars


,Pickup_point,Status,Count
0,Airport,No Cars Available,1713
1,Airport,Cancelled,198
2,City,Cancelled,1066
3,City,No Cars Available,937


Insight - Airport has more "No Cars Available" issues while City has more "Cancelled" trips. This means at Airport, drivers simply aren't present — at City, drivers are cancelling intentionally, possibly because Airport trips are long and unprofitable for return journey.

In [19]:
run_query("""
    SELECT
        Driver_id,
        COUNT(*) AS Cancellations
    FROM trips
    WHERE Status = 'Cancelled'
        AND Driver_id IS NOT NULL
    GROUP BY Driver_id
    ORDER BY Cancellations DESC
    LIMIT 10
""", title="Query 5 - Top 10 Drivers by Cancellations");


  Query 5 - Top 10 Drivers by Cancellations


,Driver_id,Cancellations
0,84.0,12
1,54.0,11
2,206.0,10
3,142.0,10
4,267.0,9
5,210.0,9
6,166.0,9
7,138.0,9
8,114.0,9
9,27.0,9


Insight - A small group of drivers are responsible for a large chunk of cancellations. These specific drivers can be flagged, warned, or given incentives to reduce cancellations — directly improving customer experience.

In [20]:
run_query("""
    SELECT
        Pickup_point,
        COUNT(*) AS Total_Requests,
        SUM(CASE WHEN Status = 'Trip Completed' THEN 1 ELSE 0 END) AS Completed,
        ROUND(
            100.0 * SUM(CASE WHEN Status = 'Trip Completed' THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS Completion_Rate_Pct
    FROM trips
    GROUP BY Pickup_point
""", title="Query 6 - Completion Rate Airport vs City");


  Query 6 - Completion Rate Airport vs City


,Pickup_point,Total_Requests,Completed,Completion_Rate_Pct
0,Airport,3238,1327,41.0
1,City,3507,1504,42.9


Insight - City has a higher completion rate than Airport — confirming that the Airport route is the bigger problem area. Uber needs to specifically focus on increasing driver availability at the Airport, especially during Night and Early Morning slots.

In [21]:
run_query("""
    SELECT
        Pickup_point,
        Time_of_Day,
        COUNT(*) AS Problem_Requests
    FROM trips
    WHERE Status IN ('Cancelled', 'No Cars Available')
    GROUP BY Pickup_point, Time_of_Day
    ORDER BY Problem_Requests DESC
    LIMIT 10
""", title="Query 7 - Worst Pickup Point and Time Slot Combos");


  Query 7 - Worst Pickup Point and Time Slot Combos


,Pickup_point,Time_of_Day,Problem_Requests
0,Airport,Evening,1145
1,City,Early Morning,962
2,Airport,Late Night,421
3,City,Morning,389
4,City,Night,214
5,City,Afternoon,205
6,Airport,Night,150
7,City,Late Night,127
8,Airport,Afternoon,114
9,City,Evening,106


Insight - Airport + Night is the single worst combination with the most unfulfilled requests — this is the core problem Uber needs to solve. Followed by Airport + Evening and City + Morning. These 3 combos should be the top priority for Uber's operations team.